# DeepGuard — DF40 Drive-native extraction

Colab's local `/content` disk is too small for DF40 plus extraction. This notebook deliberately keeps the archive and extracted dataset on Google Drive. It never requires 110 GB of local scratch space.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, os, json, time
ROOT=Path('/content/drive/MyDrive/DeepGuard')
DF40=ROOT/'datasets/DF40'
ARCH=ROOT/'datasets/archives'
DF40.mkdir(parents=True,exist_ok=True); ARCH.mkdir(parents=True,exist_ok=True)
usage=shutil.disk_usage('/content/drive')
print('Drive total:',round(usage.total/1e12,2),'TB')
print('Drive free :',round(usage.free/1e12,2),'TB')
print('DF40 target:',DF40)


## 1. Put the officially obtained DF40 archive on Drive

Do not download a large archive into `/content`. Place the official DF40 archive directly in `DeepGuard/datasets/archives/`. The exact archive/file name can vary with the DF40 distribution.


In [ ]:
archives=list(ARCH.glob('*'))
print('Archives found:')
for p in archives:
    if p.is_file(): print(' ',p.name, round(p.stat().st_size/1e9,2),'GB')
if not archives: print('NO ARCHIVE FOUND — upload/copy the official DF40 archive to:',ARCH)


## 2. Extract directly on Google Drive

This avoids the previous `Reserve at least 110 GB` error. It is slower than local SSD because Google Drive is mounted storage, but it uses your 2 TB Drive instead of Colab's small ephemeral disk.


In [ ]:
import tarfile, zipfile, pathlib
archive=next((p for p in archives if p.suffix.lower() in ['.zip','.tar','.gz','.tgz','.bz2','.xz']),None)
if archive is None:
    print('No supported archive selected yet.')
else:
    print('Selected:',archive)
    if archive.suffix.lower()=='.zip':
        with zipfile.ZipFile(archive) as z:
            z.extractall(DF40)
    else:
        with tarfile.open(archive,'r:*') as t:
            t.extractall(DF40, filter='data')
    print('Extraction complete:',DF40)


In [ ]:
files=sum(1 for p in DF40.rglob('*') if p.is_file())
size=sum(p.stat().st_size for p in DF40.rglob('*') if p.is_file())
print('Files:',files)
print('Extracted size:',round(size/1e9,2),'GB')
(DF40/'_deepguard_extraction.json').write_text(json.dumps({'timestamp':time.time(),'files':files,'bytes':size},indent=2))


### Important
If the official DF40 distribution is supplied as many files rather than one archive, copy those files directly into `datasets/DF40/` and skip extraction. Do not use `/content/DF40`.
